In [1]:
from torchmetrics.image import MultiScaleStructuralSimilarityIndexMeasure
from microssim import MicroMS3IM, MicroSSIM
from tifffile import imread, TiffFile
import os
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

micros_ms3im = MicroMS3IM()

results_list_MMSE = []
results_list_Ind = []
results_list_MMSE_Ind = []
perceptual = []
results_path = "/group/jug/Anirban/Datasets/DataForMicroSSIM/data/"
image_files = sorted([f for f in os.listdir(results_path) if f.endswith('.tif')])
print("Number of images:", len(image_files))

Number of images: 3


In [2]:
full = []
for image_file in tqdm(image_files, leave=False):
    result_file = os.path.join(results_path, image_file)
    with TiffFile(result_file) as tif:
        image = tif.asarray()
    
    full.append(image)
    
    print(result_file, image.shape)

full = np.array(full)
print("Full shape:", full.shape)

/group/jug/Anirban/Datasets/DataForMicroSSIM/data/results_summary_img0.tif (10, 1200, 1200)
/group/jug/Anirban/Datasets/DataForMicroSSIM/data/results_summary_img1.tif (10, 1200, 1200)
/group/jug/Anirban/Datasets/DataForMicroSSIM/data/results_summary_img2.tif (10, 1200, 1200)
Full shape: (3, 10, 1200, 1200)


In [3]:
gts = full[:, 0:1, :, :]
gts = np.repeat(gts, 9, axis=1)
print("Repeated GT shape:", gts.shape)
preds = full[:, 1:, :, :]
print("Predictions shape:", preds.shape)



Repeated GT shape: (3, 9, 1200, 1200)
Predictions shape: (3, 9, 1200, 1200)


In [4]:
collapsed_gts = gts.reshape(-1, gts.shape[-2], gts.shape[-1])
collapsed_preds = preds.reshape(-1, preds.shape[-2], preds.shape[-1])
print("Collapsed GT shape:", collapsed_gts.shape)
print("Collapsed Predictions shape:", collapsed_preds.shape)

Collapsed GT shape: (27, 1200, 1200)
Collapsed Predictions shape: (27, 1200, 1200)


In [5]:
microssim = MicroSSIM()
microssim.fit(collapsed_gts, collapsed_preds) # fit the parameters
micros_ms3im.fit(collapsed_gts, collapsed_preds) #similar to MultiScaleStructuralSimilarityIndexMeasure


100%|██████████| 27/27 [00:01<00:00, 20.05it/s]


In [6]:
micro_ssim_scores = []  
for i in range(gts.shape[0]):
    for j in range(gts.shape[1]):

        gt =   gts[i, j, :, :]
        pred = preds[i, j, :, :]

        score = microssim.score(gts[i,j], preds[i,j]) # score a single pair
        print(f"MicroSSIM ({i,j}): {score}")    
        micro_ssim_scores.append(score)

    average_micro_ssim = np.sum(micro_ssim_scores) / len(micro_ssim_scores)
    std_micro_ssim = np.std(np.stack(micro_ssim_scores))
    print(f"Average MicroSSIM: {average_micro_ssim}, Std: {std_micro_ssim}")
    


MicroSSIM ((0, 0)): 0.6439490433587498
MicroSSIM ((0, 1)): 0.7502313862099637
MicroSSIM ((0, 2)): 0.777546949179563
MicroSSIM ((0, 3)): 0.7224143733285574
MicroSSIM ((0, 4)): 0.7109575543730112
MicroSSIM ((0, 5)): 0.8045720384931737
MicroSSIM ((0, 6)): 0.7572168032701082
MicroSSIM ((0, 7)): 0.7362829307655091
MicroSSIM ((0, 8)): 0.7596677383708597
Average MicroSSIM: 0.740315424149944, Std: 0.043151756004398384
MicroSSIM ((1, 0)): 0.49673859630822587
MicroSSIM ((1, 1)): 0.6785016651047306
MicroSSIM ((1, 2)): 0.7127587578426574
MicroSSIM ((1, 3)): 0.6202337343099793
MicroSSIM ((1, 4)): 0.6069111629243027
MicroSSIM ((1, 5)): 0.7343881178413179
MicroSSIM ((1, 6)): 0.660047783216831
MicroSSIM ((1, 7)): 0.6163072747720096
MicroSSIM ((1, 8)): 0.6529790713286179
Average MicroSSIM: 0.691205832277676, Std: 0.07414591762901462
MicroSSIM ((2, 0)): 0.6227624686616288
MicroSSIM ((2, 1)): 0.6189123528786927
MicroSSIM ((2, 2)): 0.630448893582387
MicroSSIM ((2, 3)): 0.5411770760533913
MicroSSIM ((2, 4)

In [7]:
micro3_ssim_scores = []
for i in range(gts.shape[0]):
    for j in range(gts.shape[1]):

        micro3ssim_score = micros_ms3im.score(gts[i,j], preds[i,j])
        print(f"MicroMS3IM ({i,j}): {micro3ssim_score}")
        micro3_ssim_scores.append(micro3ssim_score)
        
    average_micro3_ssim = np.sum(micro3_ssim_scores) / len(micro3_ssim_scores)
    std_micro3_ssim = np.std(np.stack(micro3_ssim_scores))
    print(f"Average MicroMS3IM: {average_micro3_ssim}, Std: {std_micro3_ssim}")

MicroMS3IM ((0, 0)): 0.9168099164962769
MicroMS3IM ((0, 1)): 0.9439711570739746
MicroMS3IM ((0, 2)): 0.9490143656730652
MicroMS3IM ((0, 3)): 0.9244260787963867
MicroMS3IM ((0, 4)): 0.9296213388442993
MicroMS3IM ((0, 5)): 0.9491167664527893
MicroMS3IM ((0, 6)): 0.9253553748130798
MicroMS3IM ((0, 7)): 0.9318197965621948
MicroMS3IM ((0, 8)): 0.9384641647338867
Average MicroMS3IM: 0.9342887666490343, Std: 0.010844171047210693
MicroMS3IM ((1, 0)): 0.8744894862174988
MicroMS3IM ((1, 1)): 0.9254323244094849
MicroMS3IM ((1, 2)): 0.9329360723495483
MicroMS3IM ((1, 3)): 0.8895354866981506
MicroMS3IM ((1, 4)): 0.8948819637298584
MicroMS3IM ((1, 5)): 0.9298191070556641
MicroMS3IM ((1, 6)): 0.8843392729759216
MicroMS3IM ((1, 7)): 0.8938976526260376
MicroMS3IM ((1, 8)): 0.9038784503936768
Average MicroMS3IM: 0.9187671873304579, Std: 0.022366801276803017
MicroMS3IM ((2, 0)): 0.8891820907592773
MicroMS3IM ((2, 1)): 0.8976871371269226
MicroMS3IM ((2, 2)): 0.9032254219055176
MicroMS3IM ((2, 3)): 0.85423